# Real Estate Data Analysis and Model Training

This notebook demonstrates how to use the modules from the `immob/api_immobiliare` package for:
1. Loading and preprocessing real estate data
2. Feature engineering using advanced techniques
3. Training machine learning models with hyperparameter optimization
4. Model evaluation and visualization
5. Model persistence and registry
6. Interactive data visualization with Streamlit

The notebook shows the entire workflow from data retrieval to model deployment and visualization.

Author: Lucas P  
Date: July 6, 2025

## 1. Setup and Imports

First, let's import the necessary modules and packages. We'll be using:

- `sys` and `os` for path manipulation
- `pandas` for data manipulation
- `numpy` for numerical operations
- `matplotlib` and `seaborn` for static visualizations
- `logging` for improved error handling
- Our custom modules from `immob/api_immobiliare`

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from datetime import datetime
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Configure better visualization
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add the parent directory to sys.path so we can import our modules
module_path = str(Path.cwd().parent)
if module_path not in sys.path:
    sys.path.append(module_path)

# Display versions
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Verify we can import our custom modules
try:
    from data_manager import RealEstateDataManager
    from retrievers import RealEstateAdRetriever, ImmobiliareAdRetriever
    import advanced_features
    import model_utils
    import visualizations
    print("\n✅ Successfully imported custom modules!")
except ImportError as e:
    print(f"\n❌ Error importing custom modules: {e}")
    print("Make sure you're running this notebook from the correct directory.")

## 2. Loading and Preprocessing Real Estate Data

In this section, we'll demonstrate how to:

1. Load existing real estate data using the `RealEstateDataManager`
2. Create a mock retriever for testing purposes
3. Clean and preprocess the data for analysis and modeling

We'll start by initializing our data manager and loading some sample data. If you don't have existing data, we'll show how to create mock data for testing.

In [ ]:
# Initialize the data manager
data_manager = RealEstateDataManager(
    db_path="real_estate.db",
    csv_path="real_estate_data.csv",
    json_path="real_estate_data.json"
)

# Define data directory
data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)

# Check if we have existing CSV data
csv_path = data_dir / "real_estate_data.csv"

# Function to create mock data if needed
def create_mock_data(num_samples=500):
    """Create synthetic real estate data for testing"""
    logger.info(f"Creating {num_samples} mock real estate listings")
    
    # Create a mock retriever
    mock_retriever = RealEstateAdRetriever.create_mock_retriever(
        num_samples=num_samples, 
        include_random_features=True
    )
    
    # Retrieve mock data
    mock_ads = mock_retriever.retrieve_ads(max_results=num_samples)
    
    # Save to CSV
    df = pd.DataFrame([ad.to_dict() for ad in mock_ads])
    df.to_csv(csv_path, index=False)
    logger.info(f"Saved {len(df)} mock listings to {csv_path}")
    return df

# Load or create data
if csv_path.exists():
    logger.info(f"Loading existing data from {csv_path}")
    df = pd.read_csv(csv_path)
else:
    logger.info("No existing data found. Creating mock data...")
    df = create_mock_data(500)

# Display basic info about the data
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Data cleaning and preprocessing
def preprocess_data(df):
    """Clean and preprocess real estate data"""
    logger.info("Preprocessing real estate data")
    
    # Make a copy to avoid modifying the original
    processed_df = df.copy()
    
    # Handle missing values
    for col in processed_df.columns:
        missing = processed_df[col].isna().sum()
        if missing > 0:
            missing_pct = missing / len(processed_df) * 100
            logger.info(f"Column {col} has {missing} missing values ({missing_pct:.2f}%)")
            
            # For numeric columns with less than 30% missing, fill with median
            if pd.api.types.is_numeric_dtype(processed_df[col]) and missing_pct < 30:
                processed_df[col] = processed_df[col].fillna(processed_df[col].median())
                logger.info(f"  - Filled with median: {processed_df[col].median()}")
            
            # For categorical columns with less than 30% missing, fill with mode
            elif pd.api.types.is_object_dtype(processed_df[col]) and missing_pct < 30:
                mode_value = processed_df[col].mode().iloc[0]
                processed_df[col] = processed_df[col].fillna(mode_value)
                logger.info(f"  - Filled with mode: {mode_value}")
    
    # Handle price column if it exists
    if 'price' in processed_df.columns:
        # Remove outliers (prices more than 3 std devs from mean)
        mean_price = processed_df['price'].mean()
        std_price = processed_df['price'].std()
        lower_bound = mean_price - 3 * std_price
        upper_bound = mean_price + 3 * std_price
        
        outliers = (processed_df['price'] < lower_bound) | (processed_df['price'] > upper_bound)
        logger.info(f"Removing {outliers.sum()} price outliers")
        processed_df = processed_df[~outliers]
    
    # Convert date columns to datetime
    date_cols = [col for col in processed_df.columns if 'date' in col.lower()]
    for col in date_cols:
        try:
            processed_df[col] = pd.to_datetime(processed_df[col])
            logger.info(f"Converted {col} to datetime")
        except:
            logger.warning(f"Could not convert {col} to datetime")
    
    return processed_df

# Apply preprocessing
cleaned_df = preprocess_data(df)

# Show the results of preprocessing
print(f"Original shape: {df.shape}")
print(f"After preprocessing: {cleaned_df.shape}")
print("\nMissing values summary:")
print(cleaned_df.isna().sum().sort_values(ascending=False).head(10))

# Display some basic statistics
cleaned_df.describe()

## 3. Advanced Feature Engineering

Now that we have clean data, let's use our `advanced_features` module to create sophisticated features for our machine learning models. The module provides several feature engineering classes:

1. **Text-based feature extraction**: Extracts features from text descriptions using TF-IDF and Word2Vec
2. **Geospatial features**: Creates features based on location data
3. **Temporal features**: Extracts features from date and time data
4. **Image features**: Extracts features from property images (if available)

Let's apply these techniques to our dataset.

In [ ]:
# Text Feature Extraction
# Check if we have description text in our data
description_col = None
for col in cleaned_df.columns:
    if 'desc' in col.lower() or 'description' in col.lower():
        description_col = col
        break

if description_col and not cleaned_df[description_col].isna().all():
    # Initialize the text feature extractor
    from advanced_features import DescriptionFeatures
    
    text_features = DescriptionFeatures(
        language='italian',  # Change if your data is in another language
        max_features=50,     # Number of TF-IDF features to extract
        use_word2vec=True    # Enable Word2Vec features
    )
    
    # Extract text features
    logger.info(f"Extracting text features from {description_col}")
    text_feature_df = text_features.extract_features(cleaned_df[description_col])
    
    # Display the first few text features
    print(f"Generated {text_feature_df.shape[1]} text features")
    text_feature_df.head()
else:
    logger.warning("No description column found or all values are missing")
    text_feature_df = pd.DataFrame(index=cleaned_df.index)

In [ ]:
# Geospatial Feature Engineering
# Check if we have coordinate data
lat_col = None
lon_col = None
for col in cleaned_df.columns:
    if 'lat' in col.lower():
        lat_col = col
    if 'lon' in col.lower() or 'lng' in col.lower():
        lon_col = col

if lat_col and lon_col and not (cleaned_df[lat_col].isna().all() or cleaned_df[lon_col].isna().all()):
    # Initialize the location feature extractor
    from advanced_features import LocationFeatures
    
    # Define some points of interest (POIs) for Rome
    # Replace with actual coordinates for your area
    pois = {
        'city_center': (41.902782, 12.496365),  # Rome city center
        'colosseum': (41.8902, 12.4924),        # Colosseum
        'termini_station': (41.9010, 12.5013)   # Termini Station
    }
    
    location_features = LocationFeatures(
        pois=pois,
        use_clustering=True,
        n_clusters=5
    )
    
    # Extract location features
    logger.info(f"Extracting location features from {lat_col} and {lon_col}")
    location_feature_df = location_features.extract_features(
        cleaned_df[lat_col],
        cleaned_df[lon_col]
    )
    
    # Display the first few location features
    print(f"Generated {location_feature_df.shape[1]} location features")
    location_feature_df.head()
else:
    logger.warning("No coordinate columns found or all values are missing")
    location_feature_df = pd.DataFrame(index=cleaned_df.index)

In [ ]:
# Temporal Feature Engineering
# Check if we have date columns
date_cols = [col for col in cleaned_df.columns 
            if 'date' in col.lower() and pd.api.types.is_datetime64_dtype(cleaned_df[col])]

if date_cols:
    # Initialize the temporal feature extractor
    from advanced_features import TemporalFeatures
    
    date_col = date_cols[0]  # Use the first date column found
    
    temporal_features = TemporalFeatures(
        reference_date=datetime.now(),
        create_cyclical=True,
        create_age_features=True
    )
    
    # Extract temporal features
    logger.info(f"Extracting temporal features from {date_col}")
    temporal_feature_df = temporal_features.extract_features(cleaned_df[date_col])
    
    # Display the first few temporal features
    print(f"Generated {temporal_feature_df.shape[1]} temporal features")
    temporal_feature_df.head()
else:
    logger.warning("No date columns found with datetime dtype")
    temporal_feature_df = pd.DataFrame(index=cleaned_df.index)

In [ ]:
# Combine all features into a single dataframe
def prepare_model_data(df, text_features=None, location_features=None, temporal_features=None):
    """Combine basic features with engineered features"""
    logger.info("Preparing final dataset for modeling")
    
    # Start with the basic numeric features
    numeric_cols = [col for col in df.columns 
                    if pd.api.types.is_numeric_dtype(df[col]) 
                    and col != 'price']  # Exclude price as it's our target
    
    model_df = df[numeric_cols].copy()
    logger.info(f"Selected {len(numeric_cols)} basic numeric features")
    
    # Add engineered features if available
    feature_dfs = []
    
    if text_features is not None and not text_features.empty:
        feature_dfs.append(text_features)
    
    if location_features is not None and not location_features.empty:
        feature_dfs.append(location_features)
        
    if temporal_features is not None and not temporal_features.empty:
        feature_dfs.append(temporal_features)
    
    # Combine all features
    for i, feature_df in enumerate(feature_dfs):
        logger.info(f"Adding feature set {i+1} with {feature_df.shape[1]} features")
        model_df = pd.concat([model_df, feature_df], axis=1)
    
    logger.info(f"Final dataset has {model_df.shape[1]} features and {model_df.shape[0]} samples")
    
    return model_df

# Combine all features
model_data = prepare_model_data(
    cleaned_df,
    text_features=text_feature_df,
    location_features=location_feature_df,
    temporal_features=temporal_feature_df
)

# Display the final feature set
print(f"Final dataset shape: {model_data.shape}")
model_data.head()

## 4. Model Training with Hyperparameter Optimization

Now that we've prepared our features, we can train a machine learning model to predict real estate prices. We'll use the `model_utils` module to:

1. Split the data into training and testing sets
2. Create a modeling pipeline with preprocessing steps
3. Perform hyperparameter optimization
4. Train the final model with the best parameters
5. Evaluate the model performance

Let's start by preparing our data for modeling.

In [ ]:
# Prepare data for modeling
def prepare_train_test_data(features_df, target_df, test_size=0.2, random_state=42):
    """Split data into training and test sets"""
    logger.info(f"Splitting data with test_size={test_size}")
    
    X_train, X_test, y_train, y_test = train_test_split(
        features_df, target_df, test_size=test_size, random_state=random_state
    )
    
    logger.info(f"Training set: {X_train.shape[0]} samples")
    logger.info(f"Test set: {X_test.shape[0]} samples")
    
    return X_train, X_test, y_train, y_test

# Check if we have price data (our target variable)
if 'price' in cleaned_df.columns:
    # Get features and target
    X = model_data
    y = cleaned_df['price']
    
    # Split the data
    X_train, X_test, y_train, y_test = prepare_train_test_data(X, y)
    
    # Display data shapes
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")
else:
    logger.error("No 'price' column found in the data. Cannot train a price prediction model.")
    X_train, X_test, y_train, y_test = None, None, None, None

In [ ]:
# Create a modeling pipeline with preprocessing steps
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler

# Check if we have data to work with
if X_train is not None and y_train is not None:
    # Create the pipeline
    model_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestRegressor(random_state=42))
    ])
    
    # Get default parameter grid for random forest
    param_grid = model_utils.get_default_param_grid('random_forest')
    
    # Initialize the optimizer
    optimizer = model_utils.HyperparameterOptimizer(
        pipeline=model_pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    # Perform grid search
    logger.info("Starting hyperparameter optimization with grid search")
    optimization_results = optimizer.grid_search(X_train, y_train)
    
    # Display the best parameters
    print("Best parameters found:")
    for param, value in optimizer.best_params_.items():
        print(f"  {param}: {value}")
    
    # Create a final model with the best parameters
    best_model = optimizer.best_estimator_
    
    # Save optimization results
    results_dir = Path("./results")
    results_dir.mkdir(exist_ok=True)
    optimizer.save_results(results_dir / "optimization_results.json")
    logger.info("Saved optimization results")
else:
    logger.warning("No data available for model training")

## 5. Model Evaluation and Visualization of Results

Now that we have trained our model with optimized hyperparameters, let's evaluate its performance on the test set and visualize the results. We'll use:

1. Standard regression metrics (RMSE, MAE, R²)
2. Residual plots
3. Feature importance visualization
4. Prediction vs. actual price comparison

These evaluations will help us understand how well our model performs and which features are most important for predicting real estate prices.

In [ ]:
# Model evaluation
if 'best_model' in locals():
    logger.info("Evaluating model on test set")
    
    # Make predictions
    y_pred = best_model.predict(X_test)
    
    # Calculate standard metrics
    evaluation = model_utils.evaluate_model(
        model=best_model,
        X_test=X_test,
        y_test=y_test,
        feature_names=X_test.columns.tolist()
    )
    
    # Display metrics
    print("\nModel Performance Metrics:")
    for metric, value in evaluation['metrics'].items():
        print(f"  {metric}: {value:.4f}")
    
    # Create a dataframe with actual and predicted values
    results_df = pd.DataFrame({
        'actual': y_test,
        'predicted': y_pred
    })
    
    # Calculate residuals
    results_df['residual'] = results_df['actual'] - results_df['predicted']
    
    # Plot actual vs predicted values
    plt.figure(figsize=(10, 8))
    plt.scatter(results_df['actual'], results_df['predicted'], alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual Price')
    plt.ylabel('Predicted Price')
    plt.title('Actual vs. Predicted Prices')
    plt.grid(True)
    plt.show()
    
    # Plot residuals
    plt.figure(figsize=(10, 6))
    plt.scatter(results_df['predicted'], results_df['residual'], alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Price')
    plt.ylabel('Residual')
    plt.title('Residual Plot')
    plt.grid(True)
    plt.show()
    
    # Plot residual distribution
    plt.figure(figsize=(10, 6))
    plt.hist(results_df['residual'], bins=30, alpha=0.7)
    plt.xlabel('Residual')
    plt.ylabel('Frequency')
    plt.title('Residual Distribution')
    plt.grid(True)
    plt.show()
    
    # Plot feature importance if available
    if evaluation['feature_importance'] is not None:
        # Convert to dataframe for easier plotting
        importance_df = pd.DataFrame(
            list(evaluation['feature_importance'].items()),
            columns=['Feature', 'Importance']
        ).sort_values('Importance', ascending=False)
        
        # Plot top 20 features
        plt.figure(figsize=(12, 8))
        importance_df.head(20).plot(
            x='Feature', 
            y='Importance', 
            kind='bar', 
            title='Top 20 Feature Importance'
        )
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        # Display top 20 features as a table
        print("\nTop 20 Most Important Features:")
        display(importance_df.head(20))
else:
    logger.warning("No trained model available for evaluation")

## 6. Model Persistence and Registry

Now that we have a trained model with good performance, let's save it to our model registry for future use. The `model_utils` module provides a `ModelRegistry` class that allows us to:

1. Save models with metadata and performance metrics
2. Load models for prediction or further training
3. Keep track of multiple model versions
4. Find the best model based on performance metrics

Let's save our model to the registry and demonstrate how to load it back.

In [ ]:
# Save model to registry
if 'best_model' in locals() and 'evaluation' in locals():
    # Initialize model registry
    registry_dir = Path("./models")
    registry = model_utils.ModelRegistry(registry_path=registry_dir)
    logger.info(f"Initialized model registry at {registry_dir}")
    
    # Create metadata
    metadata = {
        'dataset_size': len(df),
        'train_size': len(X_train),
        'test_size': len(X_test),
        'feature_count': X_train.shape[1],
        'preprocessing_steps': [step[0] for step in model_pipeline.steps],
        'description': 'Real estate price prediction model using random forest with hyperparameter optimization',
        'model_type': 'random_forest',
        'created_by': 'Lucas P'
    }
    
    # Save model
    model_id = registry.save_model(
        model=best_model,
        model_name='price_prediction',
        feature_names=X_test.columns.tolist(),
        metrics=evaluation['metrics'],
        metadata=metadata,
        feature_importance=evaluation['feature_importance']
    )
    
    logger.info(f"Saved model to registry with ID: {model_id}")
    print(f"Model saved with ID: {model_id}")
    
    # List all models in registry
    models = registry.list_models()
    print(f"\nModels in registry: {len(models)}")
    
    # Load the saved model to demonstrate loading
    loaded_model, loaded_metadata = registry.load_model(model_id)
    logger.info(f"Successfully loaded model {model_id}")
    
    # Compare loaded model to original
    loaded_pred = loaded_model.predict(X_test)
    original_pred = best_model.predict(X_test)
    
    # Calculate MSE between original and loaded model predictions
    mse = mean_squared_error(original_pred, loaded_pred)
    print(f"MSE between original and loaded model predictions: {mse:.10f}")
    print("(Should be very close to zero, confirming successful model persistence)")
    
    # Display some of the metadata
    print("\nModel Metadata:")
    for key, value in loaded_metadata.items():
        if key not in ['feature_names', 'feature_importance']:
            print(f"  {key}: {value}")
else:
    logger.warning("No trained model available to save")

## 7. Streamlit Interactive Visualizations

Our package includes a `visualizations` module that provides interactive dashboards for real estate data using Streamlit. While we can't run Streamlit directly from a Jupyter notebook, we can:

1. Show how to call the Streamlit application
2. Preview some of the visualizations as static plots
3. Export the data for use in Streamlit

Let's create some example visualizations similar to what the Streamlit dashboard would display.

In [ ]:
# Create preview visualizations similar to what Streamlit would show
# First, let's make sure we have the cleaned data
if 'cleaned_df' in locals():
    # Price distribution visualization
    plt.figure(figsize=(12, 6))
    sns.histplot(cleaned_df['price'], kde=True, bins=50)
    plt.title('Price Distribution')
    plt.xlabel('Price')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    
    # Price by property type (if available)
    if 'type' in cleaned_df.columns:
        plt.figure(figsize=(12, 6))
        
        # Get top 10 property types by count
        top_types = cleaned_df['type'].value_counts().head(10).index
        type_data = cleaned_df[cleaned_df['type'].isin(top_types)]
        
        # Create box plot
        sns.boxplot(x='type', y='price', data=type_data)
        plt.title('Price by Property Type')
        plt.xlabel('Property Type')
        plt.ylabel('Price')
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    
    # Price by location (if available)
    location_cols = [col for col in ['city', 'district', 'neighborhood', 'zone'] if col in cleaned_df.columns]
    if location_cols:
        location_col = location_cols[0]
        plt.figure(figsize=(14, 6))
        
        # Get top 15 locations by count
        top_locations = cleaned_df[location_col].value_counts().head(15).index
        location_data = cleaned_df[cleaned_df[location_col].isin(top_locations)]
        
        # Calculate mean price by location
        location_price = location_data.groupby(location_col)['price'].mean().sort_values(ascending=False)
        
        # Create bar chart
        location_price.plot(kind='bar')
        plt.title(f'Average Price by {location_col.title()}')
        plt.xlabel(location_col.title())
        plt.ylabel('Average Price')
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    
    # Create correlation matrix visualization
    numeric_cols = [col for col in cleaned_df.columns if pd.api.types.is_numeric_dtype(cleaned_df[col])]
    if len(numeric_cols) >= 5:
        # Select top 10 numeric columns
        selected_cols = numeric_cols[:10]
        
        # Calculate correlation matrix
        corr_matrix = cleaned_df[selected_cols].corr()
        
        # Create heatmap
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(
            corr_matrix,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            square=True,
            linewidths=0.5
        )
        plt.title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.show()
    
    # Export the data for use with Streamlit
    export_path = data_dir / "streamlit_data.csv"
    cleaned_df.to_csv(export_path, index=False)
    logger.info(f"Exported data for Streamlit to {export_path}")
    print(f"\nData exported for Streamlit use to: {export_path}")
    
    # Show how to run the Streamlit app
    print("\nTo run the Streamlit visualization, execute the following command in your terminal:")
    print("streamlit run visualizations.py")
    
else:
    logger.warning("No cleaned data available for visualization")

## 8. Conclusion and Next Steps

In this notebook, we've demonstrated the complete workflow for real estate data analysis and model building using our modular package:

1. **Data Loading and Preprocessing**: We loaded real estate data and cleaned it for analysis.
2. **Advanced Feature Engineering**: We extracted features from text descriptions, geospatial data, and temporal information.
3. **Model Training with Hyperparameter Optimization**: We trained a Random Forest model and optimized its hyperparameters.
4. **Model Evaluation**: We evaluated the model using standard metrics and visualizations.
5. **Model Persistence**: We saved our trained model to a model registry for future use.
6. **Data Visualization**: We created preview visualizations similar to our Streamlit dashboard.

### Next Steps

1. **Production Deployment**: Deploy the model as an API for real-time predictions.
2. **Monitoring**: Set up monitoring for model performance and data drift.
3. **Automated Retraining**: Create a pipeline for automated retraining with new data.
4. **Feature Store**: Develop a feature store for consistent feature engineering.
5. **Enhanced Visualizations**: Add more advanced visualizations to the Streamlit dashboard.

### Resources

- Documentation: See the `docs/` directory for comprehensive documentation.
- Examples: Additional examples can be found in the `examples/` directory.
- Modules: Explore the individual modules for more functionality.

For questions or issues, please contact the project maintainers.